In [1]:
import numpy as np
import pandas as pd

## Import module
(or run %pip install -e)

In [2]:
#%pip install -e .

In [3]:
import sys
from pathlib import Path

sys.path.append(str(Path("..") / "src"))

from mobcalibrate import Calibrator
from mobcalibrate.preprocessing import knn_cluster_label
from mobcalibrate.preprocessing import compute_atus_target_table
from mobcalibrate.preprocessing import get_valid_mask_by_acs_geoid

## Prep step

In [4]:
### LOAD / PREPARE DATA

# ATUS sequences
resp = pd.read_csv('data/processed/atus_resp.csv', index_col=0)
seq = pd.read_csv('data/processed/atus_seq.csv', index_col=0)
atus_meta = pd.read_csv('data/processed/atus_meta.csv')

# ACS targets
### col target
col_margin = pd.read_csv('data/processed/age_cbsa_margins_38060.csv')
### row target
row_margin = pd.read_csv('data/processed/income_cbsa_margins_38060.csv')

### cbg-level probabilities
acs_cbg_probs = pd.read_csv('data/processed/cbg_acs_distr_38060.csv', dtype={'GEOID':str})

# Cuebiq-ATUS dist matrix
#dist = pd.read_csv('data/processed/hamming_10k_sample.csv', dtype={'GEOID':str})
dist = pd.read_parquet('data/processed/distance_matrix_3cat_full_embedding.parquet')

In [10]:
dist['20190504191857'] = dist.loc[:, '20190504191857'].str.replace('\x18', '').astype(float)
#dist.iloc[:, 1952] = dist.iloc[:, 1952].astype(float)

AttributeError: Can only use .str accessor with string values!

In [11]:
dist.iloc[:, 1952]#.astype(float)

0         0.077593
1         0.559493
2         0.029357
3         0.559493
4         0.097320
            ...   
141525    0.847517
141526    0.077593
141527    0.559493
141528    0.559493
141529    0.212296
Name: 20190504191857, Length: 141530, dtype: float64

In [12]:
i = 0
for j, dt in enumerate(dist.dtypes):
    if dt != float:
        i += 1
        print(j)
i

2033


1

In [13]:
dist['20040807040006'].mean()

0.1853510832833195

In [14]:
atus_meta.head()

,TUCASEID,TUFINLWGT,income,age,joint,cluster_label,income_cat,age_cat
0,20040707040543,20255.140632,2,2,10,2,75k_100k,45 to 66
1,20040707040558,8612.078956,2,1,9,3,75k_100k,25 to 44
2,20040807040006,13895.869568,1,2,6,0,35k_75k,45 to 66
3,20040807040542,39687.590746,1,0,4,0,35k_75k,18 to 24
4,20040807040553,2537.938345,0,2,2,3,less_than_35k,45 to 66


In [15]:
display(row_margin)
display(col_margin)

,hh_income,pop
0,less_than_35k,416253.0
1,35k_75k,547471.0
2,75k_100k,237884.0
3,greater_than_100k,543611.0


,age_group,pop
0,18 to 24,444363.0
1,25 to 44,1337601.0
2,45 to 66,1262610.0
3,67 and above,663574.0


In [16]:
acs_cbg_probs.head()

,GEOID,18 to 24,25 to 44,45 to 66,67 and above,less_than_35k,35k_75k,75k_100k,greater_than_100k
0,040130101021,0.003542,0.319953,0.279811,0.396694,0.085106,0.127660,0.182033,0.605201
1,040130101022,0.006592,0.071852,0.555043,0.366513,0.295756,0.087533,0.049072,0.567639
2,040130101023,0.023019,0.000000,0.464151,0.512830,0.149068,0.103520,0.064182,0.683230
3,040130101031,0.045374,0.129004,0.556940,0.268683,0.226510,0.275168,0.082215,0.416107
4,040130101032,0.026382,0.236809,0.499372,0.237437,0.118758,0.140351,0.170040,0.570850


In [17]:
### PREP ACS TARGET

# get total population for city
target_pop_tot = col_margin['pop'].sum() 

# get category names
row_cats = row_margin['hh_income'].values
col_cats = col_margin['age_group'].values
print(col_cats)
print(row_cats)

# normalize to get marginal distribution
col_margin = col_margin['pop'] / target_pop_tot
row_margin = row_margin['pop'] / row_margin['pop'].sum()

# set number of demographic strata and ATUS clusters
num_row_cats = len(row_cats)
num_col_cats = len(col_cats)
num_strata = num_row_cats * num_col_cats
num_clusters = 4

# set the ACS row variable name and column variable name
row_name = 'income'
col_name = 'age'


['18 to 24' '25 to 44' '45 to 66' '67 and above']
['less_than_35k' '35k_75k' '75k_100k' 'greater_than_100k']


In [18]:
### PREP ATUS TARGET

atus_target_P, joint = compute_atus_target_table(
    atus_meta, 
    stratum_var1_col='income', 
    stratum_var2_col='age', 
    cluster_label_col='cluster_label', 
    weight_col='TUFINLWGT', 
    num_var1_cats=num_row_cats,
    num_var2_cats=num_col_cats, 
    num_clusters=num_clusters
    )

atus_meta['joint'] = joint

atus_target_P

array([[0.4325988 , 0.0339982 , 0.23620993, 0.29719307],
       [0.36477615, 0.11647432, 0.32101248, 0.19773705],
       [0.3550762 , 0.22103271, 0.22186025, 0.20203083],
       [0.41028178, 0.32290664, 0.0409592 , 0.22585238],
       [0.378268  , 0.06594847, 0.25098553, 0.304798  ],
       [0.20935346, 0.07566893, 0.43650411, 0.2784735 ],
       [0.35293887, 0.10978396, 0.31108133, 0.22619583],
       [0.39777285, 0.22378025, 0.02521093, 0.35323597],
       [0.36988238, 0.20372292, 0.26291417, 0.16348054],
       [0.22849651, 0.06184007, 0.45291206, 0.25675136],
       [0.28622924, 0.07205571, 0.35554611, 0.28616895],
       [0.63952756, 0.13966989, 0.00989712, 0.21090544],
       [0.3680041 , 0.        , 0.24120701, 0.39078889],
       [0.18924382, 0.06677804, 0.50343248, 0.24054565],
       [0.26287667, 0.10937971, 0.38089003, 0.2468536 ],
       [0.44594546, 0.20819012, 0.07414072, 0.2717237 ]])

In [19]:
### PREP MOBILITY DATA


# Assign ATUS cluster label to mobility users based on knn assignment
k = 10
thres = 1/2
dist_matrix_arr = dist.values[:, :len(atus_meta)] # make sure this array only contains distances and no metadata
assigned_labels = knn_cluster_label(k, dist_matrix_arr, atus_meta['cluster_label'], thres)
dist['assigned_label'] = assigned_labels


# Filter for users whose home GEOIDs are supported in acs_cbg_probs
# (Only calibrate users whose home cbgs exist in the prepared ACS data)
geoid_col = 'GEOID'
mask, _ = get_valid_mask_by_acs_geoid(dist, acs_cbg_probs, geoid_col=geoid_col)

# (!) USE THE FILTERED USERS GOING FORWARD FROM THIS POINT (!)
#### (reset index here to ensure alignment)
users_df = dist.loc[mask].reset_index(drop=True).copy()
users_df['assigned_label_orig'] = users_df['assigned_label']
print(len(users_df[users_df['assigned_label'] == -1]))

# (!) Unassign users whose distance
"""
dist_to_medoid_99pct = {
    1: (1437, 0.28682192293378117),
    3: (6, 0.07594190845573826),
    0: (42, 0.2819953947940581),
    2: (1862, 0.0)
}
"""

dist_to_medoid_99pct = {
    2: (1437, 0.28682192293378117),
    3: (6, 0.07594190845573826),
    0: (42, 0.2819953947940581),
    1: (1022, 0.0)
}


for label, (medoid_idx, thres) in dist_to_medoid_99pct.items():
    mask2 = (users_df['assigned_label'] == label) & (users_df.iloc[:, medoid_idx] > thres)
    users_df.loc[mask2, 'assigned_label'] = -1
print(len(users_df[users_df['assigned_label'] == -1]))

# (!) Use this to initialize the Calibrator
home_cbgs = users_df[geoid_col].values
assigned_labels = users_df['assigned_label'].values

35151 rows have user GEOID missing from ACS GEOID (or null), e.g. ['040136103001' '040130506062' '040138171001' '040130405172'
 '040138118002']
Unique missing GEOIDs (excluding null): 504
81
5826


In [20]:
users_df['assigned_label'].value_counts() / len(users_df)

assigned_label
 2    0.319574
 3    0.224377
 0    0.213303
 1    0.187979
-1    0.054766
Name: count, dtype: float64

### SETTINGS

In [11]:
# number of replicates
num_replicates = 160

# set seed number for reproducibility
seed = 1234


# set geoid column name
# (name should be common across all data used here)
geoid_col = 'GEOID'

## Calibration step

In [12]:
### INITIALIZE CALIBRATOR OBJECT

calibrator = Calibrator(
    home_cbgs = home_cbgs,
    assigned_cluster_labels = assigned_labels,
    acs_cbg_probs_df = acs_cbg_probs,
    acs_row_var_name = row_name,
    acs_col_var_name = col_name,
    acs_row_cats = row_cats,
    acs_col_cats = col_cats,
    acs_row_margin = row_margin,
    acs_col_margin = col_margin,
    atus_target_table = atus_target_P,
    target_pop_tot = target_pop_tot,
    geoid_col = geoid_col,
    num_replicates = num_replicates,
    seed = seed
)

In [13]:
### CREATE MAIN AND REPLICATE WEIGHTS

calibrator.create_main_weights()
calibrator.create_replicate_weights()

In [14]:
### GET MAIN WEIGHTS

calibrator.get_main_weights()

,weight1,weight2,sampled_income_code,sampled_age_code,sampled_joint_stratum_code
0,578,1587,1,3,7
1,578,1587,1,3,7
2,466,635,3,3,15
3,466,635,3,3,15
4,490,156,2,3,11
...,...,...,...,...,...
7044,470,253,2,2,10
7045,486,295,2,1,9
7046,574,415,1,1,5
7047,601,426,1,0,4


In [15]:
# Option: return numpy array instead of pandas df for faster indexing
calibrator.get_main_weights(return_df=False)

array([[ 578, 1587,    1,    3,    7],
       [ 578, 1587,    1,    3,    7],
       [ 466,  635,    3,    3,   15],
       ...,
       [ 574,  415,    1,    1,    5],
       [ 601,  426,    1,    0,    4],
       [ 616,  942,    0,    3,    3]])

In [16]:
# use get_replicate_weights() when interested in getting the rest of the replicates (multidimensional array)
print(calibrator.get_replicate_weights(return_df=False).shape)

# use the first array dimension to index the replicate number
# e.g. calibrator.get_replicate_weights()[0] returns the first replicate (replicate_id = 1)
calibrator.get_replicate_weights()[0]

(160, 7049, 5)


array([[ 469,  287,    3,    3,   15],
       [ 504, 2025,    2,    3,   11],
       [ 589,  735,    1,    3,    7],
       ...,
       [ 590,  297,    0,    0,    0],
       [ 625,  597,    0,    1,    1],
       [ 625,  597,    0,    1,    1]])

In [17]:
# This returns False because get_replicate_weights skips main weight (replicate_id = 0)
calibrator.get_replicate_weights(return_df=True)[0].equals(calibrator.get_main_weights())

## DON'T RECOMMEND TO SET return_df = True FOR VERY LARGE DATA & BIG NUMBER OF REPLCIATES, 
## probably better to work with numpy array and index as appropriate

False

In [18]:
# This returns True because the first array/df from get_all_weights is the main weights
calibrator.get_all_weights(return_df=True)[0].equals(calibrator.get_main_weights())

True

In [19]:
# use this to get root seed number as well as spawn_key for each replicate
# calibrator.get_seed_info()